# Hesseflux Dynamic U-star and Gap Filling

This tutorial uses the stage-1 fixture to estimate dynamic u-star thresholds and then run the Python-native hesseflux backend.


In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name != "mia-tutorials" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

DATA = ROOT / "data"
REPOS = ROOT.parent
ROOT

from miaproc.eddy import (
    HessefluxConfig,
    estimate_dynamic_ustar_thresholds,
    load_stage1,
    postproc,
)

eddy_dir = DATA / "miaproc_eddy"
stage1 = load_stage1(
    path_full_output=eddy_dir / "full_output",
    path_biomet=eddy_dir / "biomet",
    tz_in="UTC",
    tz_out="UTC",
    skip_full_output=0,
    skip_biomet=0,
    drop_rain_rows=True,
)
stage1.shape


## Estimate dynamic u-star thresholds

The helper is fast and useful for explaining the diagnostics before launching the full backend.


In [ ]:
ustar = estimate_dynamic_ustar_thresholds(
    stage1,
    ustar_probs=(0.05, 0.5, 0.95),
    ustar_scenario="U50",
    ustar_min_night_samples=100,
)

print("method:", ustar.method)
print("night samples:", ustar.night_sample_count)
print("selected:", ustar.selected_scenario, ustar.selected_threshold)
ustar.thresholds_by_scenario


## Configure hesseflux

`HessefluxConfig` records both u-star policy and partitioning choices. The full `postproc` cell below usually takes a few seconds on the fixture.


In [ ]:
cfg = HessefluxConfig(
    ustar_mode="dynamic",
    ustar_probs=(0.05, 0.5, 0.95),
    ustar_scenario="U50",
    ustar_min_night_samples=100,
    partition_method="lasslop",
    swthr=20.0,
)
cfg


In [ ]:
out = postproc(stage1, engine="hesseflux", hesseflux_config=cfg)

diag = out.attrs.get("miaproc_diagnostics", {})
print("output shape:", out.shape)
print("backend:", diag.get("backend"))
print("ustar:", diag.get("ustar"))
print("partitioning:", diag.get("partitioning"))
out[["DateTime", "NEE", "NEE_f", "GPP", "Reco"]].head()


## Minimal fixed-threshold comparison

Fixed mode is the default legacy-compatible mode. Comparing diagnostics makes the policy difference explicit.


In [ ]:
fixed = postproc(stage1, engine="hesseflux", hesseflux_config=HessefluxConfig(ustar_fixed=0.1))
fixed.attrs["miaproc_diagnostics"]["ustar"]
